## Computting Committor on 2d potentials

We study transitions from a set $A$ to another set $B$ under the Brownian dynamics in $\mathbb{R}^2$:

\begin{equation*}
  dX_t = -\nabla V(X_t)\,dt + \sqrt{2\beta^{-1}} dB_t, \quad t \ge 0 \,,
\end{equation*}
where $V: \mathbb{R}^2\rightarrow \mathbb{R}$ is a potential.

The committor is the probability that $X_t$ hits $B$ before it hits $A$, i.e.

$$
  q(x) = P(\tau_{B, x} < \tau_{A,x})\,, \quad x \in \mathbb{R}^d\,,
$$

where $\tau_{A, x}, \tau_{B, x}$ are the first hitting time of sets $A$ and $B$, respectively, conditioning on the fact that the current state is $x$.     

This notebook illustrates how to compute the committor $q$ by training a neural network using trajectory data of the process.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.cm as cm
import math as math

### Potential $V$

Let us define the potential $V$ of the Brownian dynamics.

In this notebook, we consider two potentials. 

The first one has been used in previous notebooks.

In [ ]:
# Mueller-Brown potential $V(x)$ in 2d
class MuellerPotential:
    def __init__(self, *argv):
        
        # Parameters in the definition of V
        self.a = [-1, -1, -6.5, 0.7]
        self.b = [0, 0, 11, 0.6]
        self.c = [-10, -10, -6.5, 0.7]
        self.A = [-200, -100, -170, 15]
        self.xc = [1, 0, -0.5, -1]
        self.yc = [0, 0.5, 1.5, 1]

        self.x_domain = [-1.8, 1.2]
        self.y_domain = [-0.5, 2.2]
        self.v_min_max = [-130, 20]
        self.contour_levels = [-130, -100, -80, -60, -40, -20, 0.0]
        self.density_max = 0.35
        
    # the potential    
    def V(self, x):
        s = 0
        for i in range(4):
            dx = x[0] - self.xc[i]
            dy = x[1] - self.yc[i]
            s += self.A[i] * np.exp(self.a[i] * dx**2 + self.b[i] * dx * dy + self.c[i] * dy**2)
        return s
    
    # gradient of the potential    
    def gradV(self, x):
        s = 0
        dVx = 0
        dVy = 0
        for i in range(4):
            dx = x[0] - self.xc[i]
            dy = x[1] - self.yc[i]            
            dVx += self.A[i] * (2 * self.a[i] * dx + self.b[i] * dy) * np.exp(self.a[i] * dx**2 + self.b[i] * dx * dy + self.c[i] * dy**2)
            dVy += self.A[i] * (self.b[i] * dx + 2 * self.c[i] * dy) * np.exp(self.a[i] * dx**2 + self.b[i] * dx * dy + self.c[i] * dy**2)
        return np.array((dVx, dVy))

The second potential has two local minimum points, connected by a curved transition pathway. 

In [ ]:
class CurvedChannel: 
    def __init__(self, *argv):
        self.dim = 2
        self.x_domain = [-3, 3.0]
        self.y_domain = [-3, 3.0]
        self.x0 = [-1, 0]
        self.v_min_max = [-3,5]
        self.name = 'curved channel'
        self.contour_levels = [-3.0, -2.0, -1.0, 0, 1.5, 2.0, 3.0]

        self.min_A = [-1.0, 0]
        self.min_B = [1.0, 0]

        self.density_max = 0.1

        self.eps = 0.2

    def V(self, X):
        tmp1 = (X[0]**4 + X[1]**4) / 15.0 
        tmp2 = 4.0 * math.exp(-0.5 * (X[0]+2)**2 - 3.0 * (X[1])**2)
        tmp3 = 4.0 * math.exp(-0.5 * (X[0]-2)**2 - 3.0 * (X[1])**2)
        tmp4 = 2.0 * math.exp(-1.0 / self.eps * (X[0]**2/4 + X[1] * 0.5 - 1)**2) * (math.tanh(X[1]-0.2) + 1) * 0.5
        tmp5 = 6.0 * math.exp(-0.5 * X[0]**2 - 0.2 * (X[1]+1.0)**2)
        tmp6 = 0.5 * math.exp(-1.0 * X[0]**2 - 1.0 * (X[1]-2.0)**2)
        s = tmp1 - tmp2 - tmp3 - tmp4 + tmp5 + tmp6

        return s
        
    def gradV(self, X):

        tmp1 = (X[0]**4 + X[1]**4) / 15.0 
        tmp2 = math.exp(-0.5 * (X[0]+2)**2 - 3.0 * (X[1])**2)
        tmp3 = math.exp(-0.5 * (X[0]-2)**2 - 3.0 * (X[1])**2)
        tmp4 = math.exp(-1.0 / self.eps * (X[0]**2/4 + X[1] * 0.5 - 1)**2) 
        tmp5 = math.exp(-0.5 * X[0]**2 - 0.2 * (X[1]+1.0)**2)
        tmp6 = math.exp(-1.0 * X[0]**2 - 1.0 * (X[1]-2.0)**2)

        dVx = 4 * X[0]**3 / 15 + 4.0 * tmp2 * (X[0]+2) + 4.0 * tmp3 * (X[0]-2) \
              + 2.0 / self.eps * (X[0]**2/4 + X[1] * 0.5 - 1) * X[0] * tmp4 * (math.tanh(X[1]-0.2) + 1) * 0.5\
              - 6.0 * X[0] * tmp5 - 1.0 * X[0] * tmp6 
        
        dVy = 4 * X[1]**3 / 15 + 24 * tmp2 * X[1] + 24 * tmp3 * X[1] \
              - 1.0 * tmp4 * (1-math.tanh(X[1]-0.2)**2) \
              + 2.0 / self.eps * (X[0]**2/4 + X[1] * 0.5 - 1) * tmp4 * (math.tanh(X[1]-0.2) + 1) * 0.5 \
              - 2.4 * (X[1] + 1.0) * tmp5 - 1.0 * (X[1]-2.0) * tmp6

        return np.array((dVx, dVy))

### Sampling SDE

The trajectory data will be used as training data to learn the committor.

Let us write a short function for sampling the trajectory of Brownian dynamics.

In [ ]:
# sample the SDE using Euler-Maruyama scheme

def sample(pot, beta=1.0, delta_t = 0.001, N=10000, save=100, seed=42):
    
    rng = np.random.default_rng(seed=seed)
     
    X = [-0.6, 1.2]
    dim = 2 
    traj = []
    tlist = []
    for i in tqdm(range(N)):
        b = rng.normal(size=(dim,))
        X = X - pot.gradV(X) * delta_t + np.sqrt(2 * delta_t/beta) * b
        
        if i % save==0:   # store data every several steps
            traj.append(X)
            tlist.append(i * delta_t)

    return np.array(tlist), np.array(traj)

### Neural networks for committor

The committor is represented by neural networks.

Note that, since the committor takes values in $[0,1]$, we add a sigmoid function in the output layer.

In [ ]:

class Committor(nn.Module):
    def __init__(self):
        super(Committor, self).__init__()
        
        self.net = nn.Sequential(
            nn.Linear(2, 128),
            nn.Tanh(),
            nn.Linear(128, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        output = self.net(x)
        return output

### learn committor by training neural networks. 

### Recall:  

1. the committor $q$ satisfies the PDE
\begin{equation*}
\begin{aligned}
  &\mathcal{L}q = 0, \quad \mbox{on}~ (A\cup B)^c\,\\
  &q|_{\partial A} = 0, \quad q|_{\partial B} = 1\,,
\end{aligned}
  \end{equation*}
  where $\mathcal{L} = -\nabla V\cdot \nabla + \frac{1}{\beta} \Delta$ is the generator.
  
2. $q$ also solves the minimization problem
      \begin{equation*}
	\min_{f}  \Big(\frac{1}{\beta} \int_{(A\cup B)^c} |\nabla f(x)|^2 \pi(x) dx\Big) 
      \end{equation*}
      among all $C^1$-smooth $f: (A\cup B)^c\rightarrow \mathbb{R}$ such that $f|_{\partial A} = 0$ and $f|_{\partial B} = 1$.

### Loss:

We define the loss function based on the objective of the minimization problem above:

  \begin{equation*}
    \mathrm{Loss}(q) = \frac{1}{\beta N}\sum_{n=1}^N |\nabla q(X_n)|^2
    \mathbb{1}_{(A\cup B)^c}(X_n) 
    + \frac{\lambda_1}{N} \sum_{n=1}^N \Big(|q(X_n)|^2 \mathbb{1}_{A}(X_n)\Big) +
    \frac{\lambda_2}{N} \sum_{n=1}^N \Big(|q(X_n)-1|^2 \mathbb{1}_{B}(X_n) \Big)\,.
  \end{equation*}
where $\lambda_1,\lambda_2>0$ are tunable parameters.

The second and the third terms in the loss are penalty terms to impose boundary conditions.

In [ ]:
def training(model, X_A, X_B, X_other, n_epochs=200, batch_size=128, lr=0.001):
    
    train_losses = []

    optimizer = optim.Adam(model.parameters(), lr=lr)

    # note that we do mini-batch only for data outside of A and B
    train_loader = DataLoader(X_other, batch_size=batch_size, shuffle=True)

    # total number of states 
    N = X_A.shape[0] + X_B.shape[0] + X_other.shape[0]    
    
    for epoch in tqdm(range(n_epochs)):  

        model.train()
        epoch_train_loss = 0

        # Train on minibatches
        for batch_data in train_loader:

            optimizer.zero_grad()

            # we need to derivative of q wrt data. 
            batch_data.requires_grad_()

            # evaluate model on mini-batch
            q = model(batch_data)

            # compute gradient of q wrt to input x.
            g_grad = torch.autograd.grad(outputs=q.sum(), inputs=batch_data, retain_graph=True)[0]

            # first term in the loss
            loss_1 = 1.0 / (beta * N) * (g_grad**2).sum() 

            # second term in the loss: q is zero on A.
            loss_2 = (model(X_A)**2).mean()

            # third term in the loss: q is one on B.
            loss_3 = ((model(X_B)-1.0)**2).mean()

            # total loss
            loss = loss_1 + loss_2 + loss_3

            loss.backward()
            optimizer.step()
            epoch_train_loss += loss.item() * batch_data.size(0)  # Accumulate loss

        # Compute average training loss for the epoch
        epoch_train_loss /= len(train_loader.dataset)

        # Store losses for plotting
        train_losses.append(epoch_train_loss)

    # Plot training and validation loss curves
    plt.figure(figsize=(6, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss Over Epochs")
    plt.legend()
    plt.show()

### Experiment 1: 1st Potential

Define an object of the potential class

In [ ]:
pot = MuellerPotential()  

### get training data by simulating a long trajectory

In [ ]:
# constant in the SDE
beta = 0.1
# number of sampling steps.
N = 2000000

# sampling. 
# by default, the state is stored every 100 steps.
tlist, trajectory = sample(pot, beta=beta, delta_t=0.0002, N=N)

print ('shape of the trajectory data:', trajectory.shape)

### visualize the trajectory data

In [ ]:
fig = plt.figure(figsize=(12,3))

ax1 = fig.add_subplot(1, 3, 1)
ax2 = fig.add_subplot(1, 3, 2)
ax3 = fig.add_subplot(1, 3, 3)

nx = ny = 200

dx = (pot.x_domain[1] - pot.x_domain[0]) / nx
dy = (pot.y_domain[1] - pot.y_domain[0]) / ny
gridx = np.linspace(pot.x_domain[0], pot.x_domain[1], nx)
gridy = np.linspace(pot.y_domain[0], pot.y_domain[1], ny)
x_plot = np.outer(gridx, np.ones(ny)) 
y_plot = np.outer(gridy, np.ones(nx)).T 

# get grid points
x2d = np.concatenate((x_plot.reshape(nx * ny, 1), y_plot.reshape(nx * ny, 1)), axis=1)

# evaluate potential on grid points
pot_on_grid = np.array([pot.V(x) for x in x2d]).reshape(nx, ny)
# plot contour lines of the potential
contours = ax1.contour(x_plot, y_plot, pot_on_grid, levels=pot.contour_levels, cmap='coolwarm')

# scatter plot of the trajectory data
ax1.scatter(trajectory[:,0], trajectory[:,1], alpha=0.5, c='k', s=4)

ax1.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax1.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax1.set_title('trajectory')

# plot time-series of the x component
ax2.plot(tlist, trajectory[:,0])
ax2.set_ylim([pot.x_domain[0], pot.x_domain[1]])
ax2.set_title('x coodinate along trajectory')

# plot time-series of the y component
ax3.plot(tlist, trajectory[:,1])
ax3.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax3.set_title('y coodinate along trajectory')

plt.show()

We need to define the two subsets $A$ and $B$.

The trajectory data is splitted into three parts. 

1. data in $A$
2. data in $B$
3. data in $(A\cup B)^c$.

In [ ]:
# set A
def data_in_A(x):
    ac = [-0.6, 1.2]
    idx = ((x-ac)**2).sum(axis=1) < 0.2
    return idx, x[idx,:]

# set B
def data_in_B(x):
    bc = [0.6, -0.1]
    idx = ((x-bc)**2).sum(axis=1) < 0.3
    return idx, x[idx,:]

# states in A
idx_A, X_A = data_in_A(trajectory)

# states in B
idx_B, X_B = data_in_B(trajectory)

# find indices of states that are not in A and B
idx_other = np.logical_not(np.logical_or(idx_A, idx_B))

X_other = trajectory[idx_other,:]

# plot data in different colors
plt.scatter(X_A[:,0], X_A[:,1], c='b')
plt.scatter(X_B[:,0], X_B[:,1], c='r')
plt.scatter(X_other[:,0], X_other[:,1], c='g')

# number of states in A, B, and (A\cup B)^c
print ('no. of states:', X_A.shape[0], X_B.shape[0], X_other.shape[0])

plt.xlim([pot.x_domain[0], pot.x_domain[1]])
plt.ylim([pot.y_domain[0], pot.y_domain[1]])

# convert to torch tensors
X_A = torch.tensor(X_A, dtype=torch.float32)
X_B = torch.tensor(X_B, dtype=torch.float32)
X_other = torch.tensor(X_other, dtype=torch.float32)

### Training

now, let's train the committor

In [ ]:
# define the neural network
model = Committor()
# training
training(model, X_A, X_B, X_other)

### visualize the learned committor

In [ ]:
fig = plt.figure(figsize=(12,4))

ax0 = fig.add_subplot(1, 3, 1)
ax1 = fig.add_subplot(1, 3, 2)
ax2 = fig.add_subplot(1, 3, 3)

nx = ny = 200

dx = (pot.x_domain[1] - pot.x_domain[0]) / nx
dy = (pot.y_domain[1] - pot.y_domain[0]) / ny
gridx = np.linspace(pot.x_domain[0], pot.x_domain[1], nx)
gridy = np.linspace(pot.y_domain[0], pot.y_domain[1], ny)
x_plot = np.outer(gridx, np.ones(ny)) 
y_plot = np.outer(gridy, np.ones(nx)).T 

x2d = np.concatenate((x_plot.reshape(nx * ny, 1), y_plot.reshape(nx * ny, 1)), axis=1)

pot_on_grid = np.array([pot.V(x) for x in x2d]).reshape(nx, ny)

# plot the potential and its contour lines
im = ax0.pcolormesh(x_plot, y_plot, pot_on_grid, cmap='coolwarm', vmin=pot.v_min_max[0], vmax=pot.v_min_max[1])
contours = ax0.contour(x_plot, y_plot, pot_on_grid,  pot.contour_levels)
ax0.clabel(contours, inline=True, fontsize=13,colors='black')

ax0.set_aspect('equal')
ax0.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax0.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax0.set_title("Meuller-Brown potential",fontsize=15)

with torch.no_grad():
    grid_tensor = torch.tensor(x2d, dtype=torch.float32)
    # evaluate encoder on grid points
    q_values = model(grid_tensor).numpy().reshape(nx, ny)
    
    X = torch.tensor(trajectory, dtype=torch.float32)
    
    q_values_traj = model(X).numpy()

im = ax1.pcolormesh(x_plot, y_plot, q_values, cmap='coolwarm', vmin=0, vmax=1)
contours = ax1.contour(x_plot, y_plot, q_values,  10)
ax1.clabel(contours, inline=True, fontsize=13,colors='black')

# show trajectory data    
ax1.scatter(trajectory[:,0], trajectory[:,1], alpha=0.5, c='k', s=4)

ax1.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax1.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax1.set_aspect('equal')
ax1.set_title("Committor",fontsize=15)

ax2.scatter(trajectory[:,0], trajectory[:,1], alpha=0.5, c=q_values_traj, s=4, cmap='coolwarm', vmin=0, vmax=1)
ax2.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax2.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax2.set_aspect('equal')
ax2.set_title("Committor along trajectory",fontsize=15)

plt.show()

### Experiment 1: 2nd Potential

We study the second potential.

Define an object of the potential class

In [ ]:
pot = CurvedChannel()

### Sampling trajectory

To prepare training data, let's generate a long trajectory. 

In [ ]:
beta = 1.8
N = 1000000

tlist, trajectory = sample(pot, beta=beta, delta_t=0.01, N=N)

print ('shape of the trajectory data:', trajectory.shape)

### Visualize the trajectory data

In [ ]:
fig = plt.figure(figsize=(6,6))

ax1 = fig.add_subplot(1, 1, 1)

nx = ny = 200

dx = (pot.x_domain[1] - pot.x_domain[0]) / nx
dy = (pot.y_domain[1] - pot.y_domain[0]) / ny
gridx = np.linspace(pot.x_domain[0], pot.x_domain[1], nx)
gridy = np.linspace(pot.y_domain[0], pot.y_domain[1], ny)
x_plot = np.outer(gridx, np.ones(ny)) 
y_plot = np.outer(gridy, np.ones(nx)).T 

# get grid points
x2d = np.concatenate((x_plot.reshape(nx * ny, 1), y_plot.reshape(nx * ny, 1)), axis=1)

# evaluate potential on grid points
pot_on_grid = np.array([pot.V(x) for x in x2d]).reshape(nx, ny)
# plot contour lines of the potential
contours = ax1.contour(x_plot, y_plot, pot_on_grid, levels=pot.contour_levels, cmap='coolwarm')

# plot the potential and its contour lines
im = ax1.pcolormesh(x_plot, y_plot, pot_on_grid, cmap='coolwarm', vmin=pot.v_min_max[0], vmax=pot.v_min_max[1])
contours = ax1.contour(x_plot, y_plot, pot_on_grid,  pot.contour_levels)
ax1.clabel(contours, inline=True, fontsize=13,colors='black')

# scatter plot of the trajectory data
ax1.scatter(trajectory[:,0], trajectory[:,1], alpha=0.5, c='k', s=4)

ax1.set_aspect('equal')
ax1.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax1.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax1.set_title('trajectory',fontsize=15)

plt.show()

###  data splitting 

Based on the plot above, we define the two sets $A$ and $B$.

The trajectory data is splitted into:

1. states in A
2. states in B
3. states outsides A and B.


In [ ]:
# set A
def data_in_A(x):
    ac = [-2, 0]
    idx = ((x-ac)**2).sum(axis=1) < 1
    return idx, x[idx,:]

# set B
def data_in_B(x):
    bc = [2, 0]
    idx = ((x-bc)**2).sum(axis=1) < 1
    return idx, x[idx,:]

# find states in A
idx_A, X_A = data_in_A(trajectory)

# find states in B
idx_B, X_B = data_in_B(trajectory)

# indices of states outsides A and B
idx_other = np.logical_not(np.logical_or(idx_A, idx_B))
X_other = trajectory[idx_other,:]

# plot states in different sets using different colors
plt.scatter(X_A[:,0], X_A[:,1], c='b')
plt.scatter(X_B[:,0], X_B[:,1], c='r')
plt.scatter(X_other[:,0], X_other[:,1], c='g')

# number of states in A, B, and (A\cup B)^c
print ('no. of states:', X_A.shape[0], X_B.shape[0], X_other.shape[0])

plt.xlim([pot.x_domain[0], pot.x_domain[1]])
plt.ylim([pot.y_domain[0], pot.y_domain[1]])

# convert to torch tensors
X_A = torch.tensor(X_A, dtype=torch.float32)
X_B = torch.tensor(X_B, dtype=torch.float32)
X_other = torch.tensor(X_other, dtype=torch.float32)

### learn the committor by training neural networks.

In [ ]:
# define the neural network
model = Committor()
# training
training(model, X_A, X_B, X_other)

### visualize the learned committor

In [ ]:
fig = plt.figure(figsize=(12,4))

ax0 = fig.add_subplot(1, 3, 1)
ax1 = fig.add_subplot(1, 3, 2)
ax2 = fig.add_subplot(1, 3, 3)

nx = ny = 200

dx = (pot.x_domain[1] - pot.x_domain[0]) / nx
dy = (pot.y_domain[1] - pot.y_domain[0]) / ny
gridx = np.linspace(pot.x_domain[0], pot.x_domain[1], nx)
gridy = np.linspace(pot.y_domain[0], pot.y_domain[1], ny)
x_plot = np.outer(gridx, np.ones(ny)) 
y_plot = np.outer(gridy, np.ones(nx)).T 

x2d = np.concatenate((x_plot.reshape(nx * ny, 1), y_plot.reshape(nx * ny, 1)), axis=1)

pot_on_grid = np.array([pot.V(x) for x in x2d]).reshape(nx, ny)

# plot the potential and its contour lines
im = ax0.pcolormesh(x_plot, y_plot, pot_on_grid, cmap='coolwarm', vmin=pot.v_min_max[0], vmax=pot.v_min_max[1])
contours = ax0.contour(x_plot, y_plot, pot_on_grid,  pot.contour_levels)
ax0.clabel(contours, inline=True, fontsize=13,colors='black')

ax0.set_aspect('equal')
ax0.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax0.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax0.set_title("Potential",fontsize=15)

with torch.no_grad():
    grid_tensor = torch.tensor(x2d, dtype=torch.float32)
    # evaluate encoder on grid points
    q_values = model(grid_tensor).numpy().reshape(nx, ny)
    
    # committor at states along trajectory
    X = torch.tensor(trajectory, dtype=torch.float32)
    q_values_traj = model(X).numpy()
    
# show trajectory data    
ax1.scatter(trajectory[:,0], trajectory[:,1], alpha=0.5, c='k', s=4)

ax1.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax1.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax1.set_aspect('equal')
ax1.set_title("Committor",fontsize=15)

im = ax1.pcolormesh(x_plot, y_plot, q_values, cmap='coolwarm', vmin=0, vmax=1)
contours = ax1.contour(x_plot, y_plot, q_values,  10)
ax1.clabel(contours, inline=True, fontsize=13,colors='black')

ax2.scatter(trajectory[:,0], trajectory[:,1], alpha=0.5, c=q_values_traj, s=4, cmap='coolwarm', vmin=0, vmax=1)
ax2.set_xlim([pot.x_domain[0], pot.x_domain[1]])
ax2.set_ylim([pot.y_domain[0], pot.y_domain[1]])
ax2.set_aspect('equal')
ax2.set_title("Committor along trajectory",fontsize=15)

plt.show()